In [1]:
import os
import sys
import subprocess

# Colab環境で画面非表示(ヘッドレス)で最速動作させるための環境変数
os.environ["PYBULLET_GEOM_VIEWER"] = "0"

def install_and_import():
    try:
        import pybullet as p
        import pybullet_data
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pybullet"])
        import pybullet as p
        import pybullet_data
    try:
        import cv2
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "opencv-python"])
        import cv2
    try:
        import numpy as np
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
        import numpy as np
    try:
        import pandas as pd
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
        import pandas as pd

    return p, pybullet_data, cv2, np, pd

p, pybullet_data, cv2, np, pd = install_and_import()

# 長アームの長さを厳密に定義
L1 = 0.9  # 青アームの長さ
L2 = 0.8  # 緑アームの長さ

def create_robot_urdf():
    """長アームロボット（合計1.7m）のURDFファイルを動的に自動生成"""
    urdf_text = f"""<?xml version="1.0"?>
<robot name="planar_arm">
  <link name="base_link">
    <visual>
      <geometry><cylinder length="0.1" radius="0.2"/></geometry>
      <material name="gray"><color rgba="0.5 0.5 0.5 1"/></material>
    </visual>
  </link>
  <joint name="joint1" type="revolute">
    <parent link="base_link"/>
    <child link="link1"/>
    <origin xyz="0 0 0.05" rpy="0 0 0"/>
    <axis xyz="0 0 1"/>
    <limit effort="300" velocity="4.0" lower="-3.1416" upper="3.1416"/>
  </joint>
  <link name="link1">
    <visual>
      <origin xyz="{L1/2} 0 0" rpy="0 1.5707 0"/>
      <geometry><cylinder length="{L1}" radius="0.04"/></geometry>
      <material name="blue"><color rgba="0.1 0.4 0.8 1"/></material>
    </visual>
  </link>
  <joint name="joint2" type="revolute">
    <parent link="link1"/>
    <child link="link2"/>
    <origin xyz="{L1} 0 0" rpy="0 0 0"/>
    <axis xyz="0 0 1"/>
    <limit effort="300" velocity="4.0" lower="-3.1416" upper="3.1416"/>
  </joint>
  <link name="link2">
    <visual>
      <origin xyz="{L2/2} 0 0" rpy="0 1.5707 0"/>
      <geometry><cylinder length="{L2}" radius="0.03"/></geometry>
      <material name="green"><color rgba="0.1 0.8 0.4 1"/></material>
    </visual>
    <inertial>
      <mass value="1.0"/>
      <inertia ixx="0.1" ixy="0" ixz="0" iyy="0.1" iyz="0" izz="0.1"/>
    </inertial>
  </link>
  <link name="ee_link">
    <visual>
      <geometry><sphere radius="0.05"/></geometry>
      <material name="red"><color rgba="0.9 0.1 0.1 1"/></material>
    </visual>
  </link>
  <joint name="ee_joint" type="fixed">
    <parent link="link2"/>
    <child link="ee_link"/>
    <origin xyz="{L2} 0 0" rpy="0 0 0"/>
  </joint>
</robot>
"""
    with open("planar_arm.urdf", "w") as f:
        f.write(urdf_text)

def analytical_ik(x, y):
    """余弦定理を用いた2自由度平面アームの直接解析解法（IK）"""
    r_sq = x**2 + y**2
    r = np.sqrt(r_sq)

    if r > (L1 + L2):
        x = x * (L1 + L2) / r
        y = y * (L1 + L2) / r
        r_sq = x**2 + y**2
        r = L1 + L2
    elif r < abs(L1 - L2):
        r = abs(L1 - L2)
        r_sq = r**2

    cos_q2 = (r_sq - L1**2 - L2**2) / (2 * L1 * L2)
    cos_q2 = np.clip(cos_q2, -1.0, 1.0)
    q2 = np.arccos(cos_q2)

    theta = np.arctan2(y, x)
    cos_alpha = (L1**2 + r_sq - L2**2) / (2 * L1 * r)
    cos_alpha = np.clip(cos_alpha, -1.0, 1.0)
    alpha = np.arccos(cos_alpha)

    q1 = theta - alpha
    q1 = np.arctan2(np.sin(q1), np.cos(q1))
    q2 = np.arctan2(np.sin(q2), np.cos(q2))

    return [q1, q2]

def main():
    print("--- 【ver8: 長アーム×余弦定理×目標位置ノイズ混入版】開始 ---")
    create_robot_urdf()

    physicsClient = p.connect(p.DIRECT)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)

    p.loadURDF("plane.urdf")
    robot_id = p.loadURDF("planar_arm.urdf", useFixedBase=True)

    target_visual = p.createVisualShape(p.GEOM_SPHERE, radius=0.04, rgbaColor=[1, 0, 0, 1])
    target_id = p.createMultiBody(baseMass=0, baseVisualShapeIndex=target_visual, basePosition=[0.8, 0.5, 0.05])

    video_filename = "robot_pointing_ver8_noisy.mp4"
    score_filename = "ver8_noisy_scores.csv"
    fps = 24
    duration_sec = 60
    total_frames = fps * duration_sec
    img_w, img_h = 640, 480

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(video_filename, fourcc, fps, (img_w, img_h))

    dt = 1.0 / 240.0
    p.setTimeStep(dt)
    steps_per_frame = int((1.0 / fps) / dt)

    target_pos = np.array([0.8, 0.5, 0.05])
    change_target_interval = fps * 4

    target_count = 0
    current_target_min_dist = float('inf')
    all_scores = []
    score_records = []
    last15_mean, last15_var = 0.0, 0.0

    print(f"動画を生成しています... 総フレーム数: {total_frames}")

    for frame in range(total_frames):
        if frame % change_target_interval == 0:
            if frame > 0:
                all_scores.append(current_target_min_dist)
                recent_scores = all_scores[-15:]
                last15_mean = float(np.mean(recent_scores))
                last15_var = float(np.var(recent_scores)) if len(recent_scores) > 1 else 0.0

                score_records.append({
                    "Target_ID": target_count,
                    "Target_X": float(target_pos[0]),
                    "Target_Y": float(target_pos[1]),
                    "Shortest_Distance_m": float(current_target_min_dist),
                    "Last15_Mean_m": last15_mean,
                    "Last15_Variance": last15_var
                })
                print(f"[Target #{target_count} 完了] 最小距離: {current_target_min_dist:.4f}m | 直近15回平均: {last15_mean:.4f}m | 直近15回分散: {last15_var:.6f}")

            target_count += 1
            angle = np.random.uniform(-np.pi, np.pi)
            radius = np.random.uniform(0.5, 1.4)
            target_pos = np.array([radius * np.cos(angle), radius * np.sin(angle), 0.05])

            p.resetBasePositionAndOrientation(target_id, target_pos, [0, 0, 0, 1])
            current_target_min_dist = float('inf')

        for _ in range(steps_per_frame):
            ee_state = p.getLinkState(robot_id, 2)
            ee_pos = np.array(ee_state[0])
            error = target_pos[:2] - ee_pos[:2]
            dist_to_target = np.linalg.norm(error)

            if dist_to_target < current_target_min_dist:
                current_target_min_dist = dist_to_target

            # 目標値入力に5cmのガウスノイズを混入
            noisy_x = float(target_pos[0]) + np.random.normal(0, 0.05)
            noisy_y = float(target_pos[1]) + np.random.normal(0, 0.05)

            joint_poses = analytical_ik(noisy_x, noisy_y)

            p.setJointMotorControl2(bodyIndex=robot_id, jointIndex=0, controlMode=p.POSITION_CONTROL, targetPosition=joint_poses[0], maxVelocity=4.0)
            p.setJointMotorControl2(bodyIndex=robot_id, jointIndex=1, controlMode=p.POSITION_CONTROL, targetPosition=joint_poses[1], maxVelocity=4.0)

            p.stepSimulation()

        # === 【カメラ修正パッチ適用済】アスペクト比の無限発散バグを完全解消 ===
        view_matrix = p.computeViewMatrix(
            cameraEyePosition=[0.0, 0.0, 2.5],
            cameraTargetPosition=[0.0, 0.0, 0.0],
            cameraUpVector=[0.0, 1.0, 0.0]
        )
        proj_matrix = p.computeProjectionMatrixFOV(
            fov=60.0,
            aspect=1.0,
            nearVal=0.1,
            farVal=10.0
        )
        _, _, rgb_img, _, _ = p.getCameraImage(
            width=img_w,
            height=img_h,
            viewMatrix=view_matrix,
            projectionMatrix=proj_matrix,
            renderer=p.ER_TINY_RENDERER
        )

        rgb_array = np.array(rgb_img, dtype=np.uint8).reshape(img_h, img_w, 4)
        bgr_frame = cv2.cvtColor(rgb_array, cv2.COLOR_RGBA2BGR)

        cv2.putText(bgr_frame, f"Frame: {frame}/{total_frames} (Target #{target_count}) [Ver8: Noisy Input]", (15, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(bgr_frame, f"Current Dist: {dist_to_target:.3f}m", (15, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(bgr_frame, f"Target Min Dist: {current_target_min_dist:.3f}m", (15, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
        cv2.putText(bgr_frame, f"Last 15 Mean: {last15_mean:.4f}m", (15, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150, 150, 255), 1, cv2.LINE_AA)
        cv2.putText(bgr_frame, f"Last 15 Var:  {last15_var:.6f}", (15, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 150, 255), 1, cv2.LINE_AA)

        video_writer.write(bgr_frame)

    score_records.append({
        "Target_ID": target_count,
        "Target_X": float(target_pos[0]),
        "Target_Y": float(target_pos[1]),
        "Shortest_Distance_m": float(current_target_min_dist),
        "Last15_Mean_m": last15_mean,
        "Last15_Variance": last15_var
    })
    df = pd.DataFrame(score_records)
    df.to_csv(score_filename, index=False)
    video_writer.release()
    p.disconnect()
    print(f"\n--- 完了。動画: {video_filename} / スコア: {score_filename} ---")

if __name__ == "__main__":
    main()


--- 【ver8: 長アーム×余弦定理×目標位置ノイズ混入版】開始 ---
動画を生成しています... 総フレーム数: 1440
[Target #1 完了] 最小距離: 0.0012m | 直近15回平均: 0.0012m | 直近15回分散: 0.000000
[Target #2 完了] 最小距離: 0.0003m | 直近15回平均: 0.0008m | 直近15回分散: 0.000000
[Target #3 完了] 最小距離: 0.0003m | 直近15回平均: 0.0006m | 直近15回分散: 0.000000
[Target #4 完了] 最小距離: 0.0006m | 直近15回平均: 0.0006m | 直近15回分散: 0.000000
[Target #5 完了] 最小距離: 0.0007m | 直近15回平均: 0.0006m | 直近15回分散: 0.000000
[Target #6 完了] 最小距離: 0.0002m | 直近15回平均: 0.0006m | 直近15回分散: 0.000000
[Target #7 完了] 最小距離: 0.0002m | 直近15回平均: 0.0005m | 直近15回分散: 0.000000
[Target #8 完了] 最小距離: 0.0004m | 直近15回平均: 0.0005m | 直近15回分散: 0.000000
[Target #9 完了] 最小距離: 0.0002m | 直近15回平均: 0.0005m | 直近15回分散: 0.000000
[Target #10 完了] 最小距離: 0.0008m | 直近15回平均: 0.0005m | 直近15回分散: 0.000000
[Target #11 完了] 最小距離: 0.0003m | 直近15回平均: 0.0005m | 直近15回分散: 0.000000
[Target #12 完了] 最小距離: 0.0013m | 直近15回平均: 0.0005m | 直近15回分散: 0.000000
[Target #13 完了] 最小距離: 0.0012m | 直近15回平均: 0.0006m | 直近15回分散: 0.000000
[Target #14 完了] 最小距離: 0.0003m | 直近15回平均: 0.000